# Stochastic gradient descent (SGD)

_Artificial Intelligence (CS221)_

**Don't wait for the whole dataset. Step downhill after each single example.**

Gradient descent shrinks the loss by stepping downhill — but the *full* gradient looks at every example before taking one step. That's slow. **Stochastic** gradient descent takes a step after looking at just **one** example, so it starts improving almost immediately.

This notebook builds SGD from scratch, one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the end. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

We'll train a linear classifier with the **hinge loss** on a tiny, clearly separable dataset. We build it in three steps: (1) the data, (2) the SGD training loop, (3) reading the result.

### Step 1 — Build a tiny labelled dataset

Each row of `X` is one example: two coordinates plus a constant `1` in the last column. That constant is the **bias** feature — it lets the decision boundary sit anywhere instead of being forced through the origin.

`y` holds the labels, encoded as `-1` / `+1` (not `0` / `1`) because the hinge loss is defined in terms of the *signed margin* `y · (w·x)`.

In [ ]:
rng = np.random.default_rng(0)   # seeded so results are reproducible

# Each row is [x1, x2, bias]. Two clearly separated clouds of 3 points each.
X = np.array([
    [1, 1, 1],   # cloud A (label -1)
    [1, 2, 1],
    [2, 1, 1],
    [4, 4, 1],   # cloud B (label +1)
    [5, 4, 1],
    [4, 5, 1],
], dtype=float)

y = np.array([-1, -1, -1, +1, +1, +1], dtype=float)

w = np.zeros(3)   # one weight per feature (x1, x2, bias); start at zero
eta = 0.05        # learning rate: how big a step we take per update

print("X shape:", X.shape, " weights:", w)

### Step 2 — The SGD training loop

The hinge loss for one example is `max(0, 1 - margin)` where `margin = y · (w·x)`.

- If `margin >= 1`, the example is classified correctly *and* confidently → the gradient is zero → **no update**.
- If `margin < 1`, the gradient is `-y·x`, so the SGD step `w ← w - eta·gradient` becomes `w ← w + eta·y·x`.

We loop over the examples in a **random order** each epoch (the "stochastic" part) and update the weights one example at a time.

In [ ]:
for epoch in range(8):
    # Visit the 6 examples in a fresh random order every epoch.
    for i in rng.permutation(len(y)):
        margin = y[i] * (w @ X[i])      # signed margin for this one example

        if margin < 1:                  # only misclassified / low-confidence points update
            gradient = -y[i] * X[i]     # gradient of the hinge loss at this point
            w = w - eta * gradient      # the SGD step: nudge w downhill

    # After each full pass, measure the average hinge loss over all examples.
    margins = y * (X @ w)
    loss = np.mean(np.maximum(1 - margins, 0))
    print("epoch %d   hinge loss = %.3f" % (epoch, loss))

### Step 3 — Read the trained model

`w` now defines the decision boundary. We predict with `sign(w·x)`: positive → class `+1`, negative → class `-1`. Compare the predictions to `y` below: after just 8 epochs most match, but one point is still misclassified — a reminder that SGD improves the fit gradually, and a few more epochs (or a larger `eta`) would close the gap.

In [ ]:
print("final weights:", np.round(w, 3))
print("predictions  :", np.sign(X @ w).astype(int))
print("true labels  :", y.astype(int))

## Does it work on real data?

A toy dataset always separates nicely. Let's run the *same* SGD on real **breast-cancer** scans — one tumour at a time — and watch the loss fall, then plot the result.

### First, look at the data

Before we train on the breast-cancer scans, here's the full dataset — every feature column, the two class labels, and a few real rows. (The training code below uses only a couple of these columns so the result is easy to plot.)

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

### Step 1 — Load and standardise two features

We take two measurements (mean radius and mean texture) and **standardise** them (subtract the mean, divide by the std). SGD is sensitive to feature scale; standardising keeps the step size sensible across both features.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()

# Keep just two features (columns 0 and 1) so we can plot the result in 2D.
X = StandardScaler().fit_transform(data.data[:, [0, 1]])
y = (2 * data.target - 1).astype(float)   # map {0,1} labels to {-1,+1}

# Append the bias column of 1s, exactly like the toy example.
Xb = np.hstack([X, np.ones((len(X), 1))])

print("examples:", Xb.shape[0], " features (incl. bias):", Xb.shape[1])

### Step 2 — Train, tracking the loss each epoch

Same loop as before, but now over ~569 real examples for 12 epochs. We record the loss after every epoch so we can plot the learning curve.

In [ ]:
w = np.zeros(3)
eta = 0.01
losses = []

for epoch in range(12):
    for i in rng.permutation(len(y)):
        if y[i] * (w @ Xb[i]) < 1:        # same hinge condition as before
            w = w + eta * y[i] * Xb[i]    # same update, written out directly
    losses.append(np.mean(np.maximum(1 - y * (Xb @ w), 0)))

accuracy = ((Xb @ w > 0).astype(int) == data.target).mean()
print("final accuracy:", round(accuracy, 3))

### Step 3 — Visualise the learning curve and the data

Left: the hinge loss should fall and flatten as SGD converges. Right: the two tumour clouds in feature space — overlapping, which is why real accuracy is below 100%.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left — loss per epoch
ax1.plot(range(12), losses, "-o", color="#4ea1ff")
ax1.set_xlabel("epoch")
ax1.set_ylabel("mean hinge loss")
ax1.set_title("hinge loss per epoch")

# Right — the two tumour clouds
malignant = data.target == 0
benign = data.target == 1
ax2.scatter(X[malignant, 0], X[malignant, 1], c="#ff7b72", s=12, label="malignant")
ax2.scatter(X[benign, 0], X[benign, 1], c="#4ea1ff", s=12, label="benign")
ax2.set_xlabel("radius (std)")
ax2.set_ylabel("texture (std)")
ax2.set_title("two tumour clouds")
ax2.legend()

plt.tight_layout()
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Change the learning rate `eta` on the real-data run to `0.1` and to `0.001`. How does the loss curve change?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Re-run Step 2 with `eta = 0.1`, then `eta = 0.001`, replotting `losses` each time.
- With `eta = 0.1` the curve drops fast but is **jumpy** — big steps overshoot.
- With `eta = 0.001` the curve is smooth but barely moves in 12 epochs — steps are too small.

**Answer:** the learning rate trades **speed** against **stability**; `0.01` is a reasonable middle ground here.

</details>